[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/spatialft/spatialft.github.io/blob/main/notebooks/03_finetune.ipynb)

# Notebook 3 — Fine-Tuning

LoRA fine-tune LFM2-350M on StepGame spatial reasoning data.

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO = Path('/content/spatialft.github.io')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/spatialft/spatialft.github.io.git', str(REPO)], check=True)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from src.colab_utils import bootstrap_colab_repo, get_repo_paths, publish_artifacts

REPO = bootstrap_colab_repo(REPO)
PATHS = get_repo_paths(REPO)
print(f'Repo ready at {REPO}')
print(f'Using local repo storage: {PATHS["repo_root"]}')


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

SENTINEL = Path('/tmp/spatialft_notebook03_deps_v2')
if not SENTINEL.exists():
    subprocess.run([
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        '-U',
        '--upgrade-strategy',
        'only-if-needed',
        '-r',
        '../requirements.txt',
    ], check=True)
    SENTINEL.write_text('ok')
    print('Dependencies updated. Restarting the runtime once to load clean binary wheels...')
    os.kill(os.getpid(), 9)

print('Dependencies ready for notebook 03.')


In [ ]:
import json

import matplotlib.pyplot as plt
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

from src.dataset import SYSTEM_PROMPT


In [ ]:
MODEL_ID       = 'LiquidAI/LFM2-350M'
MAX_SEQ_LENGTH = 512
LORA_RANK      = 16
OUTPUT_DIR     = str(PATHS['results_root'] / 'finetuned' / 'checkpoint')


In [ ]:
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.model_max_length = MAX_SEQ_LENGTH

peft_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_RANK * 2,
    target_modules="all-linear",  # auto-detect linear layers for the base model
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(base_model, peft_config)
model.print_trainable_parameters()


In [ ]:
with open(PATHS['data_root'] / 'processed' / 'train_formatted.json') as f:
    train_data = json.load(f)

dataset = Dataset.from_list(train_data)
print(f'Training on {len(dataset)} examples')
print(dataset[0]['full_text'][:300])


In [ ]:
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="full_text",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        warmup_steps=50,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=False,  # 4-bit quant handles memory; AMP breaks on mixed bf16 base layers
        bf16=False,
        logging_steps=20,
        output_dir=OUTPUT_DIR,
        save_strategy="epoch",
        optim="paged_adamw_8bit",
        seed=42,
    ),
)

trainer.train()

loss_points = [entry for entry in trainer.state.log_history if 'loss' in entry and 'epoch' in entry]
LOSS_CURVE_PATH = PATHS['results_root'] / 'finetuned' / 'loss_curve.png'
LOSS_CURVE_PATH.parent.mkdir(parents=True, exist_ok=True)

if loss_points:
    plt.figure(figsize=(8, 4.5))
    plt.plot([entry['epoch'] for entry in loss_points], [entry['loss'] for entry in loss_points], linewidth=2)
    plt.xlabel('Epoch')
    plt.ylabel('Training Loss')
    plt.title('Fine-Tuning Loss Curve')
    plt.tight_layout()
    plt.savefig(LOSS_CURVE_PATH, dpi=150)
    plt.show()
    print(f'Saved loss curve to {LOSS_CURVE_PATH}')
else:
    raise RuntimeError('No loss values were logged during training, so loss_curve.png could not be created.')


In [ ]:
# Save LoRA adapter locally
LOCAL_ADAPTER = PATHS['adapter_dir']
LOCAL_ADAPTER.parent.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(LOCAL_ADAPTER))
tokenizer.save_pretrained(str(LOCAL_ADAPTER))
print(f'Adapter saved to {LOCAL_ADAPTER.resolve()}')


In [ ]:
publish_artifacts(
    [
        'results/finetuned/loss_curve.png',
        'results/finetuned/lora_adapter',
    ],
    'Add fine-tuning artifacts [notebook 03]',
    repo_dir=REPO,
)
